# Bring Your Container with Streaming Reponse in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your own custom agent with Streaming response in Amazon Bedrock AgentCore Runtime. This example demonstrates how to build a FastAPI using StrandsAgents adequate to Amazon Bedrock AgentCore Runtime standards and make a requests with correct processing of the streaming response. 

### Tutorial Details

|Information| Details|
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational with Streaming|
| Agent type          | Single         |
| Agentic Framework   | Strands Agents |
| LLM model           | Anthropic Claude Sonnet 4 |
| Tutorial components | Custom Strands Agent Container with Streaming responses and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Intermediate                                                                     |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3|

### Tutorial Architecture

In this tutorial we will describe how to build and deploy your own agent container into Amazon Bedrock AgentCore Runtime with streaming response.

For demonstration purposes, we will use a Strands Agent using Amazon Bedrock models with streaming capabilities.

In our example we will use a simple agent with two tools in separate python modules: `get_weather` and `get_time`, but with streaming response capabilities.

### Tutorial Key Features

* Custom Agent API Container
* Streaming responses from agents on Amazon Bedrock AgentCore Runtime
* Real-time partial result deliver
* Using Amazon Bedrock models with streaming
* Using Strands Agents and FastAPI with async streaming support

## Prerequisites

To execute this tutorial you will need:
* Python 3.11+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker or podman running

In [ ]:
#!uv add -r requirements.txt --active

## Preparing your streaming agent custom container

To get closer to a real case scenario, we will build an agent that has it's core functionality distributed in different modules. 

For this tutorial we will build a chess evaluator agent. This agent will be capable to answer general questions about chess and, given a chess position with a FEN string (FEN - Forsyth-Edwards Notation - is a standard notation for describing chess positions), it will be able to evaluate the position using chess-api.com API or search Lichess users and Masters database for statistics on the position.

We will have the agent in the `api.py` file, important configurations in the `config.py` file and the available tools in the `tools.py` file.

Let's start with the `api.py` file. Here, we will build a simple FastAPI engine that will run inside the container. Notice that:
- We are using `async def` for your entrypoint function
- We are using `yield` to stream chunks as they become available
- Clients will receive Content-Type: text/event-stream responses
- The endpoints follow Amazon Bedrock AgentCore's endpoint standards: a post endpoint `/invocations` and a get endpoint `/ping` for healthchecks.
- For streaming, the `return` statement for the POST endpoint should return a `StreamingResponse` FastAPI class.

In [ ]:
%%writefile api.py
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Dict, Any
from strands import Agent
from strands.models import BedrockModel

from config import MODEL_CONFIG, AWS_REGION, MODEL_PARAMS

from tools import (
    get_position_evaluation,
    get_masters_games,
    get_lichess_games
)


modelinstance = BedrockModel(model_id=MODEL_CONFIG["chess-agent"], region_name=AWS_REGION, **MODEL_PARAMS)

SYSTEM_PROMPT = """You are a friendly Chess AI Agent. You main goal is to help users improve at chess and have a better
understanding of the game.

To do that, you will analyze given positions, make comments on a position and answer general chess related questions from the user.

You have access to these tools:

1. get_position_evaluation: send a FEN string to get analysis information on that given position.
2. get_masters_games: get information on masters games with the same given FEN string position, get overall data, what is the opening, possible next moves and top games (from lichess database)
3. get_lichess_games: get information on lichess users games with the same given FEN string position, get overall data, what is the opening, possible next moves and top games (from lichess database)

Don't give long answers. Engage in a chatty style conversation.
"""

agent = Agent(
    model=modelinstance,
    system_prompt=SYSTEM_PROMPT,
    tools=[
        get_position_evaluation,
        get_masters_games,
        get_lichess_games
    ],
    callback_handler=None,
)

app = FastAPI(title="Chess Evaluator Agent", version="1.0.0")

class InvocationRequest(BaseModel):
    input: Dict[str, Any]


@app.post("/invocations")
async def stream_response(request: InvocationRequest):
    async def generate(agent=agent):
        try:
            async for event in agent.stream_async(request.input.get("prompt", "")):
                if "data" in event:
                    # Only stream text chunks to the client
                    yield event["data"]
        except Exception as e:
            yield f"Error: {str(e)}"

    return StreamingResponse(
        generate(),
        media_type="text/event-stream"
    )


@app.get("/ping")
async def ping():
    return {"status": "healthy"}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8080)

Next, in the `tools.py` file, we will set up functions to call the external APIs.

In [ ]:
%%writefile tools.py
import json
from strands import Agent, tool
import requests

@tool
def get_position_evaluation(fen: str) -> dict:
    response = requests.post(
        "https://chess-api.com/v1", 
        json={
            "fen": fen,
            "depth": 18
        }
    )
    return json.loads(response.text)


@tool
def get_masters_games(fen: str) -> dict:
    response = requests.get(
        "https://explorer.lichess.ovh/masters",
        params = {"fen": fen}
    )
    return json.loads(response.text)


@tool
def get_lichess_games(fen: str) -> dict:
    response = requests.get(
        "https://explorer.lichess.ovh/lichess",
        params = {"fen": fen}
    )
    return json.loads(response.text)

Next, let's put together our `config.py` file with some configurations (simplified for educational purposes.)

In [ ]:
%%writefile config.py
"""
Configuration module for the GenAI Strategy Agent system.
Contains settings for AI models and knowledge base IDs.
"""

# Default model for all agents
DEFAULT_MODEL = "us.anthropic.claude-sonnet-4-20250514-v1:0"

# Model configuration for specific agents
# Override DEFAULT_MODEL for specific agents if needed
MODEL_CONFIG = {
    "chess-agent": DEFAULT_MODEL
}

# AWS region for Bedrock models
AWS_REGION = "us-east-1"

# Model parameters
MODEL_PARAMS = {
    "temperature": 0.7,
    "max_tokens": 4096
}

Now, we will build the container for Amazon Bedrock AgentCore Runtime. For the container, notice that:
- the base image should be built using linux/arm64 architecture to be compatible with AgentCore Runtime (better performance, more efficient resources utilization).

In [ ]:
%%writefile Dockerfile
FROM --platform=linux/arm64 ghcr.io/astral-sh/uv:python3.11-bookworm-slim

WORKDIR /app

# Copy uv files
COPY pyproject.toml uv.lock ./

# Install dependencies
RUN uv sync --frozen --no-cache

# Copy agent file
COPY api.py config.py tools.py ./

# Expose port
EXPOSE 8080

CMD ["uv", "run", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8080"]

Next, we will build the container image and upload it to an ECR repository.

In [ ]:
%%writefile build.sh
#!/bin/bash
set -e

# If the ECR repository is not created, create it
aws ecr describe-repositories --repository-names "chess-agent/runtime/api" --region us-east-1 &>/dev/null || (echo "Creating ECR repository..." && aws ecr create-repository --repository-name "chess-agent/runtime/api" --region us-east-1)

# AWS Account Number - CHANGE TO YOUR ACTUAL ACCOUNT NUMBER
ACCOUNT_NUMBER="123456789101"
# ECR repository
ECR_REPO=$ACCOUNT_NUMBER".dkr.ecr.us-east-1.amazonaws.com/chess-agent/runtime/api"
# VERSION
VERSION="v1"

# Login to ECR
aws ecr get-login-password --region us-east-1 | podman login --username AWS --password-stdin $ACCOUNT_NUMBER.dkr.ecr.us-east-1.amazonaws.com

# Build the Docker image for Intel platform
podman build --platform=linux/arm64 -t chess-agent-runtime-api:$VERSION .

# Tag the image
podman tag chess-agent-runtime-api:$VERSION $ECR_REPO:$VERSION

# Push the image to ECR
podman push $ECR_REPO:$VERSION

echo "Successfully built and pushed image to $ECR_REPO:$VERSION"


In [ ]:
%%bash
sh build.sh

## Deploy AgentCore Runtime

To deploy an Amazon Bedrock AgentCore Runtime using our custom container image, we will need an IAM Role. The code below creates one.

In [ ]:
from utils import create_agentcore_role

agent_name="chess_agent_runtime_demo"
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)

Let's now deploy an Amazon Bedrock AgentCore Runtime using our custom container image.

In [ ]:
import boto3

client = boto3.client('bedrock-agentcore-control')

response = client.create_agent_runtime(
    agentRuntimeName=agent_name,
    agentRuntimeArtifact={
        'containerConfiguration': {
            'containerUri': '123456789101.dkr.ecr.us-east-1.amazonaws.com/chess-agent/runtime/api:v1'
        }
    },
    networkConfiguration={"networkMode": "PUBLIC"},
    roleArn=agentcore_iam_role['Role']['Arn']
)

print(f"Agent Runtime created successfully!")
print(f"Agent Runtime ARN: {response['agentRuntimeArn']}")
print(f"Status: {response['status']}")

### Streaming request responses

Now, we will make a request to our agent and stream its response.

Get your Agent Runtime ARN printed in the last code chunk and paste it accordingly.

In [ ]:
import boto3
import json
import uuid

agent_core_client = boto3.client('bedrock-agentcore', region_name='us-east-1')

my_prompt = "Can you briefly explain what is the Italian Opening?"

payload = json.dumps({
    "input": {"prompt": my_prompt},
})

# sessionId = uuid.uuid4()
sessionId = 'f4510a4c-3e5f-4511-a8cc-eb9ef5c8a9ad'

response = agent_core_client.invoke_agent_runtime(
    agentRuntimeArn=response['agentRuntimeArn'],
    runtimeSessionId=sessionId,
    payload=payload,
    qualifier="DEFAULT"
)

print("Streaming response:\n")

try:
    if "text/event-stream" in response.get("contentType", ""):
        streaming_body = response["response"]
        buffer = b''  # Buffer to handle incomplete UTF-8 sequences
        
        while True:
            # Use larger chunks for simpler approach but still handle UTF-8 properly
            chunk = streaming_body.read(8)  # 16 bytes - larger chunks, less frequent processing
            if not chunk:
                # End of stream - decode any remaining buffer
                if buffer:
                    try:
                        decoded_chunk = buffer.decode("utf-8")
                        print(decoded_chunk, end='', flush=True)
                    except UnicodeDecodeError:
                        pass  # Skip invalid bytes
                break
            
            buffer += chunk
            
            # Try to decode the buffer
            try:
                decoded_chunk = buffer.decode("utf-8")
                print(decoded_chunk, end='', flush=True)
                buffer = b''  # Clear buffer after successful decode
            except UnicodeDecodeError:
                # Incomplete UTF-8 sequence at end of chunk
                # Find the last complete UTF-8 character boundary
                for i in range(len(buffer) - 1, max(0, len(buffer) - 4), -1):
                    try:
                        # Try to decode up to position i
                        decoded_chunk = buffer[:i].decode("utf-8")
                        print(decoded_chunk, end='', flush=True)
                        buffer = buffer[i:]  # Keep remaining bytes for next iteration
                        break
                    except UnicodeDecodeError:
                        continue
                else:
                    # If we can't find a valid boundary, skip the first byte
                    buffer = buffer[1:]
    else:
        streaming_body = response["response"]
        data = streaming_body.read()
        print(data.decode('utf-8'))
        
except Exception as e:
    print(f"An error occurred: {e}")


*Et voilà!*. Here is your streaming response from your custom agent deployed into AgentCore Runtime. 

Play a little bit with some different chess questions or develop an agent example of your own.

## Benefits of Streaming Responses

Streaming responses provide several key advantages:

### User Experience
* **Immediate Feedback**: Users see partial results as they become available
* **Perceived Performance**: Responses feel faster even if total time is the same
* **Progressive Display**: Long responses can be displayed incrementally

### Technical Benefits
* **Memory Efficient**: Process large responses without loading everything into memory
* **Timeout Prevention**: Avoid timeouts on long-running operations
* **Real-time Processing**: Handle real-time data as it becomes available

### Use Cases
* **Content Generation**: Long-form writing, reports, documentation
* **Data Analysis**: Progressive results from complex calculations
* **Multi-step Workflows**: Show progress through complex agent reasoning
* **Real-time Monitoring**: Live updates from monitoring agents

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

iam_client = boto3.client('iam')

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId='<PASTE YOUR AGENT ID HERE>',
    
)

response = ecr_client.delete_repository(
    repositoryName='<PASTE YOUR ECR REPOSITORY NAME HERE>',
    force=True
)

policies = iam_client.list_role_policies(
    RoleName=agentcore_iam_role['Role']['RoleName'],
    MaxItems=100
)

for policy_name in policies['PolicyNames']:
    iam_client.delete_role_policy(
        RoleName=agentcore_iam_role['Role']['RoleName'],
        PolicyName=policy_name
    )
iam_response = iam_client.delete_role(
    RoleName=agentcore_iam_role['Role']['RoleName']
)

# Congratulations!

You have successfully implemented and deployed a custom streaming agent using Amazon Bedrock AgentCore Runtime! 

## What you've learned:
* How to implement streaming responses using async generators and StreamingResponse
* How to build a custom container image with several python modules and a custom agent
* How to process streaming responses on the client side
* The benefits of streaming for user experience and performance

## Next steps:
* Experiment with different streaming patterns for your use cases
* Implement custom streaming logic for complex multi-step workflows
* Explore combining streaming with other AgentCore features like Memory and Gateway
* Consider implementing client-side streaming visualization for better UX